In [1]:
using Revise

In [3]:
using GeneRegulatorySystems
using GeneRegulatorySystems.Models
using GeneRegulatorySystems.Models: Wrapped, Instant, Label, Descriptions, EmptyDescription
using GeneRegulatorySystems.Models.Scheduling
using GeneRegulatorySystems.Models.Scheduling: Primitive, Schedule as GRSSchedule
using GeneRegulatorySystems.Models.NetworkRepresentation
using GeneRegulatorySystems.Specifications
using JSON

In [4]:
# Fixed _label: handle EmptyDescription (e.g. ResampleEachBinomial)
_label(wrapped::Models.Wrapped) = _label(Models.describe(wrapped.definition))
_label(label::Models.Label) = label.label
_label(model::Models.Instant) = repr(model)
_label(::EmptyDescription) = ""
function _label(desc::Descriptions)
    i = findfirst(d -> d isa Label, desc.descriptions)
    i !== nothing ? _label(desc.descriptions[i]) : ""
end

# Fixed _validate_spec: handle Vector (top-level JSON array = List spec)
function _validate_spec(spec::AbstractDict{Symbol})::Vector{String}
    messages = String[]
    isempty(spec) && return ["ERROR: Schedule specification is empty"]
    try
        Models.Model(
            spec,
            bindings = Dict(
                :seed => get(spec, :seed, "default"),
                :into => "",
                :channel => "",
                :defaults => Models.load_defaults(),
            ),
        )
    catch e
        push!(messages, "ERROR: $(e)")
    end
    return messages
end

function _validate_spec(spec::AbstractVector)::Vector{String}
    messages = String[]
    isempty(spec) && return ["ERROR: Schedule specification is empty"]
    try
        Models.Model(
            spec,
            bindings = Dict(
                :seed => "default",
                :into => "",
                :channel => "",
                :defaults => Models.load_defaults(),
            ),
        )
    catch e
        push!(messages, "ERROR: $(e)")
    end
    return messages
end

function _seed(spec::AbstractDict{Symbol})
    get(spec, :seed, "default")
end
_seed(::AbstractVector) = "default"

function _collect_segments(grs_schedule)::Vector
    segments = []
    next_id = Ref(1)
    function dryrun_collector(primitive!, x, dt; path, _...)
        label = _label(primitive!.f!.model)
        push!(segments, (
            id = next_id[],
            execution_path = path,
            model_path = primitive!.path,
            from = x.t,
            to = x.t + (isfinite(dt) ? dt : 0.0),
            label = label,
        ))
        next_id[] += 1
    end
    grs_schedule(Models.FlatState(); dryrun = dryrun_collector)
    return segments
end

function test_reify(spec_string::String; name::String="")
    spec = JSON.parse(spec_string, dicttype=Dict{Symbol, Any})

    # Validate
    msgs = _validate_spec(spec)
    errors = filter(m -> startswith(m, "ERROR"), msgs)
    if !isempty(errors)
        return (name=name, status="VALIDATION_FAIL", error=first(errors), segments=0)
    end

    # Build schedule
    seed = _seed(spec)
    bindings = Dict(:seed => seed, :into => "", :channel => "", :defaults => Models.load_defaults())
    specification = Specifications.Specification(spec; bound = Set(keys(bindings)))
    grs_schedule = GRSSchedule(; specification, bindings)

    # Collect segments (tests _label too)
    segments = _collect_segments(grs_schedule)

    return (name=name, status="OK", error="", segments=length(segments))
end

test_reify (generic function with 1 method)

In [5]:
# Test all schedules
schedule_dir = "tools/server/storage/schedules/examples"
files = filter(f -> endswith(f, ".schedule.json"), readdir(schedule_dir))

results = []
for f in files
    name = replace(f, ".schedule.json" => "")
    spec_string = read(joinpath(schedule_dir, f), String)
    try
        r = test_reify(spec_string; name)
        push!(results, r)
        symbol = r.status == "OK" ? "+" : "x"
        println("[$symbol] $name: $(r.status) ($(r.segments) segments) $(r.error)")
    catch e
        push!(results, (name=name, status="EXCEPTION", error=string(e), segments=0))
        println("[x] $name: EXCEPTION -- $(e)")
    end
end

println("\n--- Summary ---")
ok = count(r -> r.status == "OK", results)
println("$ok / $(length(results)) passed")

[+] ACDC-simple: OK (13 segments) 
[+] ACDC: OK (17001 segments) 
[+] aggregations: OK (2 segments) 
[x] bursts: EXCEPTION -- MethodError(Main._label, (GeneRegulatorySystems.Models.Resampling.ResampleEachBinomial(0.9),), 0x000000000000988b)
[+] cascade: OK (2 segments) 
[+] channels: OK (10 segments) 
[+] copies: OK (12 segments) 
[+] coupled-repressilator: OK (505 segments) 
[x] differentiated-medium: EXCEPTION -- MethodError(Main._label, (GeneRegulatorySystems.Models.Resampling.ResampleEachBinomial(0.9),), 0x000000000000988b)
[x] differentiated-small: EXCEPTION -- MethodError(Main._label, (GeneRegulatorySystems.Models.Resampling.ResampleEachBinomial(0.9),), 0x000000000000988b)
[+] differentiation-tree: OK (12 segments) 
[+] differentiation: OK (3 segments) 
[x] extraction: EXCEPTION -- MethodError(Main._label, (GeneRegulatorySystems.Models.Resampling.ResampleTargetMeanEachBinomial(50.0),), 0x000000000000988b)
[+] kronecker-big: OK (4503 segments) 
[x] kronecker-huge: EXCEPTION -- Int